# Full DocVQA Pipeline

**End-to-End Pipeline: Image → OCR → Layout → Graph → QA Generation → CSV**

## Pipeline Stages:
1. **OCR**: PaddleOCR to extract text and bounding boxes
2. **Layout Analysis**: Group text regions, classify region types
3. **Graph Building**: Create semantic layout graph with spatial/semantic edges
4. **QA Generation**: Rule-based + LLM-based question-answer generation
5. **Export**: Save results to JSON and CSV

## Input:
- Images from `dataset/DocVQA_Images/{train,validation,test}/`

## Output:
- Graph JSONs: `output/full_pipeline/{split}/{doc_id}.json`
- QA CSV: `output/qa_csv/docvqa_qas_all_{timestamp}.csv`

## 1️. Setup & Configuration

In [1]:
import sys
import json
import pandas as pd
from pathlib import Path
from datetime import datetime
from collections import Counter
from tqdm import tqdm

# Setup paths
sys.path.insert(0, '..')
BASE = Path("../")

# ==========================================
# CONFIGURATION - ADJUST THESE VALUES
# ==========================================
CONFIG = {
    # Data paths
    "images_folder": BASE / "dataset/DocVQA_Images",
    "graph_output_folder": BASE / "output/full_pipeline",
    "csv_output_folder": BASE / "output/qa_csv",
    
    # Processing limits
    "max_images_per_split": 400,  # Max images to process per split
    "max_qas_per_doc": 10,        # Max QA pairs per document
    "splits": ["train", "validation", "test"],  # Which splits to process
    
    # Pipeline options
    "skip_existing_graphs": True,  # Skip already processed images
    "skip_existing_qas": False,    # Regenerate QAs even if exists
}

# Template paths
RULE_TEMPLATES_PATH = BASE / "src/qa/rule_based_templates.json"
LLM_TEMPLATES_PATH = BASE / "src/qa/llm_prompt_templates.json"

print("=" * 70)
print(" Full DocVQA Pipeline")
print("=" * 70)
print(f"\n Paths:")
print(f"  Images: {CONFIG['images_folder'].resolve()}")
print(f"  Graph output: {CONFIG['graph_output_folder'].resolve()}")
print(f"  CSV output: {CONFIG['csv_output_folder'].resolve()}")
print(f"\n Configuration:")
print(f"  Max images/split: {CONFIG['max_images_per_split']}")
print(f"  Max QAs/doc: {CONFIG['max_qas_per_doc']}")
print(f"  Splits: {CONFIG['splits']}")
print(f"  Skip existing graphs: {CONFIG['skip_existing_graphs']}")

 Full DocVQA Pipeline

 Paths:
  Images: D:\Thach\HUST\Project_1\VQA\code\dataset\DocVQA_Images
  Graph output: D:\Thach\HUST\Project_1\VQA\code\output\full_pipeline
  CSV output: D:\Thach\HUST\Project_1\VQA\code\output\qa_csv

 Configuration:
  Max images/split: 400
  Max QAs/doc: 10
  Splits: ['train', 'validation', 'test']
  Skip existing graphs: True


## 2️. Initialize Pipeline Components

In [2]:
# Import pipeline components
from src.ocr.ocr_processor import PaddleOCRProcessor
from src.ocr.layout_analyzer import DocumentLayoutAnalyzer
from src.graph.graph_builder import GraphBuilder
from src.utils.pipeline import FullPipelineProcessor
from src.utils.batch_processor import BatchProcessor

print("=" * 70)
print(" Initializing Pipeline Components")
print("=" * 70)

# 1. Initialize OCR Processor
print("\n[1/4] Initializing PaddleOCR...")
ocr_processor = PaddleOCRProcessor(
    use_doc_orientation_classify=False,
    use_doc_unwarping=False,
    use_textline_orientation=False
)

# 2. Initialize Layout Analyzer
print("\n[2/4] Initializing Layout Analyzer...")
layout_analyzer = DocumentLayoutAnalyzer(
    y_overlap_threshold=0.5,
    line_height_tolerance=0.3,
    max_x_gap_ratio=3.0,
    block_vertical_gap=20,
    block_x_overlap_threshold=0.3
)
print("→ Layout Analyzer ready!")

# 3. Initialize Graph Builder
print("\n[3/4] Initializing Graph Builder...")
graph_builder = GraphBuilder(
    iou_threshold=0.1,
    distance_threshold=200.0,
    projection_threshold=0.3,
    max_neighbors=5,
    min_edge_score=0.2
)
print("→ Graph Builder ready!")

# 4. Create Full Pipeline Processor
print("\n[4/4] Creating Full Pipeline Processor...")
pipeline_processor = FullPipelineProcessor(
    ocr_processor=ocr_processor,
    layout_analyzer=layout_analyzer,
    graph_builder=graph_builder
)
print("→ Pipeline Processor ready!")

# Create Batch Processor
batch_processor = BatchProcessor(pipeline_processor)

print("\n All components initialized successfully!")

 Initializing Pipeline Components

[1/4] Initializing PaddleOCR...


c:\Users\Thach\miniconda3\envs\project\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Checking connectivity to the model hosters, this may take a while. To bypass this check, set `DISABLE_MODEL_SOURCE_CHECK` to `True`.


Đang khởi tạo PaddleOCR engine...


c:\Users\Thach\miniconda3\envs\project\lib\site-packages\paddle\utils\cpp_extension\extension_utils.py:718: UserWarning: No ccache found. Please be aware that recompiling all source files may be required. You can download and install ccache from: https://github.com/ccache/ccache/blob/master/doc/INSTALL.md
  warnings.warn(warning_message)
Creating model: ('PP-OCRv5_server_det', None)
Model files already exist. Using cached files. To redownload, please delete the directory manually: `C:\Users\Thach\.paddlex\official_models\PP-OCRv5_server_det`.
Creating model: ('PP-OCRv5_server_rec', None)
Model files already exist. Using cached files. To redownload, please delete the directory manually: `C:\Users\Thach\.paddlex\official_models\PP-OCRv5_server_rec`.


-> PaddleOCR đã sẵn sàng!

[2/4] Initializing Layout Analyzer...
→ Layout Analyzer ready!

[3/4] Initializing Graph Builder...
→ Graph Builder ready!

[4/4] Creating Full Pipeline Processor...
→ Pipeline Processor ready!

 All components initialized successfully!


## 3️. Stage 1: OCR + Layout + Graph Processing

In [3]:
# Create output directories
CONFIG['graph_output_folder'].mkdir(parents=True, exist_ok=True)
CONFIG['csv_output_folder'].mkdir(parents=True, exist_ok=True)

print("=" * 70)
print(" Stage 1: OCR → Layout → Graph Processing")
print("=" * 70)

# Check available images
for split in CONFIG['splits']:
    split_folder = CONFIG['images_folder'] / split
    if split_folder.exists():
        num_images = len(list(split_folder.glob('*.png')))
        print(f"  {split}: {num_images} images available")
    else:
        print(f"  {split}: ⚠️ Folder not found")

 Stage 1: OCR → Layout → Graph Processing
  train: 39463 images available
  validation: 5349 images available
  test: 5188 images available


In [4]:
# Run batch processing for OCR → Layout → Graph
print("\n Starting batch processing...")
print(f"Processing up to {CONFIG['max_images_per_split']} images per split...")

start_time = datetime.now()

graph_stats = batch_processor.process_dataset(
    images_folder=CONFIG['images_folder'],
    output_folder=CONFIG['graph_output_folder'],
    subsets=CONFIG['splits'],
    max_images_per_subset=CONFIG['max_images_per_split'],
    skip_existing=CONFIG['skip_existing_graphs']
)

elapsed = datetime.now() - start_time

print(f"\n{'='*70}")
print(" Stage 1 Summary: OCR + Layout + Graph")
print(f"{'='*70}")
print(f"Total time: {elapsed}")
print(f"Total processed: {graph_stats['total_processed']}")
print(f"Total success: {graph_stats['total_success']}")
print(f"Total failed: {graph_stats['total_failed']}")


 Starting batch processing...
Processing up to 400 images per split...

Processing subset: TRAIN
Found 400 images


Processing train: 100%|██████████| 400/400 [29:27<00:00,  4.42s/it]



TRAIN Summary:
  ✅ Success: 400
  ❌ Failed: 0
  📊 Total: 400

Processing subset: VALIDATION
Found 400 images


Processing validation: 100%|██████████| 400/400 [31:21<00:00,  4.70s/it]



VALIDATION Summary:
  ✅ Success: 400
  ❌ Failed: 0
  📊 Total: 400

Processing subset: TEST
Found 400 images


Processing test: 100%|██████████| 400/400 [31:32<00:00,  4.73s/it]


TEST Summary:
  ✅ Success: 400
  ❌ Failed: 0
  📊 Total: 400

 Stage 1 Summary: OCR + Layout + Graph
Total time: 1:32:21.073722
Total processed: 1200
Total success: 1200
Total failed: 0


## 4️. Stage 2: QA Generation

In [5]:
# Load QA templates
import importlib
import src.qa.llm as llm_module
importlib.reload(llm_module)
from src.qa.llm import generate_qas

print("=" * 70)
print(" Stage 2: QA Generation")
print("=" * 70)

# Load templates
rule_templates = json.loads(RULE_TEMPLATES_PATH.read_text(encoding="utf-8"))
llm_templates = json.loads(LLM_TEMPLATES_PATH.read_text(encoding="utf-8"))

print(f" Rule templates loaded: {len(rule_templates.get('templates', []))} templates")
print(f" LLM templates loaded: {len(llm_templates.get('prompt_templates', {}))} prompt types")

 Stage 2: QA Generation
 Rule templates loaded: 12 templates
 LLM templates loaded: 5 prompt types


In [6]:
def generate_qas_for_dataset(splits, max_docs_per_split, max_qas_per_doc):
    """
    Generate QA pairs for all processed documents.
    
    Returns:
        all_records: List of QA records for CSV
        stats: Statistics dictionary
    """
    all_records = []
    stats = {split: {"processed": 0, "total_qas": 0, "errors": 0} for split in splits}
    
    for split in splits:
        split_dir = CONFIG['graph_output_folder'] / split
        if not split_dir.exists():
            print(f" Split folder not found: {split_dir}")
            continue
        
        # Get processed graph JSONs
        doc_files = sorted(split_dir.glob("*.json"))[:max_docs_per_split]
        
        print(f"\n{'='*60}")
        print(f" Generating QAs for {split} ({len(doc_files)} documents)")
        print("=" * 60)
        
        for doc_path in tqdm(doc_files, desc=f"{split}"):
            try:
                # Load document graph
                doc_data = json.loads(doc_path.read_text(encoding="utf-8"))
                doc_id = doc_path.stem
                
                # Get source image name
                src_image = doc_data.get("metadata", {}).get("source_image", "")
                
                # Generate QA pairs
                qas = generate_qas(doc_data, rule_templates, llm_templates, max_qas=max_qas_per_doc)
                
                # Add each QA as a record
                for qa_idx, qa in enumerate(qas):
                    record = {
                        "split": split,
                        "doc_id": doc_id,
                        "image_file": src_image,
                        "qa_id": f"{doc_id}_{qa_idx}",
                        "question": qa.get("question", ""),
                        "answer": qa.get("answer", ""),
                        "reasoning_type": qa.get("reasoning_type", ""),
                        "reasoning_explanation": qa.get("reasoning_explanation", ""),
                        "evidence_region_ids": json.dumps(qa.get("evidence_region_ids", [])),
                        "evidence_quotes": json.dumps(qa.get("evidence_quotes", []), ensure_ascii=False),
                    }
                    all_records.append(record)
                
                stats[split]["processed"] += 1
                stats[split]["total_qas"] += len(qas)
                
            except Exception as e:
                stats[split]["errors"] += 1
                # Uncomment for debugging:
                # print(f"\n Error processing {doc_path.name}: {e}")
                continue
    
    return all_records, stats

print("\n Starting QA generation...")
start_time = datetime.now()

all_records, qa_stats = generate_qas_for_dataset(
    CONFIG['splits'],
    CONFIG['max_images_per_split'],
    CONFIG['max_qas_per_doc']
)

elapsed = datetime.now() - start_time
print(f"\n QA generation completed in {elapsed}")


 Starting QA generation...

 Generating QAs for train (400 documents)


train: 100%|██████████| 400/400 [00:10<00:00, 36.49it/s]



 Generating QAs for validation (400 documents)


validation: 100%|██████████| 400/400 [00:04<00:00, 85.15it/s]



 Generating QAs for test (400 documents)


test: 100%|██████████| 400/400 [00:05<00:00, 75.56it/s]


 QA generation completed in 0:00:20.985851


In [7]:
# Display QA Statistics
print("=" * 70)
print(" Stage 2 Summary: QA Generation")
print("=" * 70)

total_docs = sum(s["processed"] for s in qa_stats.values())
total_qas = sum(s["total_qas"] for s in qa_stats.values())
total_errors = sum(s["errors"] for s in qa_stats.values())

print(f"\n Overall Summary:")
print(f"  - Total documents processed: {total_docs}")
print(f"  - Total QA pairs generated: {total_qas}")
print(f"  - Average QAs per document: {total_qas/total_docs:.2f}" if total_docs > 0 else "  - N/A")
print(f"  - Errors: {total_errors}")

print(f"\n Per-Split Statistics:")
for split, s in qa_stats.items():
    avg = s['total_qas']/s['processed'] if s['processed'] > 0 else 0
    print(f"  {split:12s}: {s['processed']:4d} docs, {s['total_qas']:5d} QAs (avg: {avg:.1f})")

# Create DataFrame
df = pd.DataFrame(all_records)
print(f"\n DataFrame shape: {df.shape}")

# Reasoning type distribution
if len(df) > 0:
    print(f"\n Reasoning Type Distribution:")
    reasoning_counts = df['reasoning_type'].value_counts()
    for rtype, count in reasoning_counts.items():
        pct = count / len(df) * 100
        print(f"  - {rtype:30s}: {count:5d} ({pct:5.1f}%)")

 Stage 2 Summary: QA Generation

 Overall Summary:
  - Total documents processed: 1200
  - Total QA pairs generated: 1926
  - Average QAs per document: 1.60
  - Errors: 0

 Per-Split Statistics:
  train       :  400 docs,   612 QAs (avg: 1.5)
  validation  :  400 docs,   711 QAs (avg: 1.8)
  test        :  400 docs,   603 QAs (avg: 1.5)

 DataFrame shape: (1926, 10)

 Reasoning Type Distribution:
  - lookup                        :  1561 ( 81.0%)
  - comprehension                 :   365 ( 19.0%)


## 5️. Stage 3: Export to CSV

In [8]:
# Export to CSV
print("=" * 70)
print(" Stage 3: Export to CSV")
print("=" * 70)

timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")

# # Save combined CSV (all splits)
# combined_csv_path = CONFIG['csv_output_folder'] / f"docvqa_qas_all_{timestamp}.csv"
# df.to_csv(combined_csv_path, index=False, encoding="utf-8-sig")
# print(f"\n✅ Saved combined CSV:")
# print(f"   {combined_csv_path}")
# print(f"   → {len(df)} records")

 Stage 3: Export to CSV


In [9]:
# Save full statistics JSON
stats_output = {
    "timestamp": timestamp,
    "pipeline_version": "1.0.0",
    "config": {
        "max_images_per_split": CONFIG['max_images_per_split'],
        "max_qas_per_doc": CONFIG['max_qas_per_doc'],
        "splits": CONFIG['splits'],
        "skip_existing_graphs": CONFIG['skip_existing_graphs']
    },
    "graph_processing": {
        "total_processed": graph_stats['total_processed'],
        "total_success": graph_stats['total_success'],
        "total_failed": graph_stats['total_failed'],
        "by_split": graph_stats['by_subset']
    },
    "qa_generation": {
        "total_documents": total_docs,
        "total_qa_pairs": total_qas,
        "total_errors": total_errors,
        "avg_qas_per_doc": round(total_qas/total_docs, 2) if total_docs > 0 else 0,
        "per_split": {split: dict(s) for split, s in qa_stats.items()},
        "reasoning_type_distribution": reasoning_counts.to_dict() if len(df) > 0 else {}
    }
}

# Export to single file: finally_dataset.csv
final_csv_path = CONFIG['csv_output_folder'] / "finally_dataset.csv"
df.to_csv(final_csv_path, index=False, encoding="utf-8-sig")

print("=" * 70)
print(" Final Dataset Exported")
print("=" * 70)
print(f" File: {final_csv_path.resolve()}")
print(f" Total records: {len(df)}")
print(f"   - Train: {len(df[df['split']=='train'])}")
print(f"   - Validation: {len(df[df['split']=='validation'])}")
print(f"   - Test: {len(df[df['split']=='test'])}")

 Final Dataset Exported
 File: D:\Thach\HUST\Project_1\VQA\code\output\qa_csv\finally_dataset.csv
 Total records: 1926
   - Train: 612
   - Validation: 711
   - Test: 603


## 6️. Preview & Validation

In [10]:
# Preview CSV Output
print("=" * 70)
print(" CSV Preview (Sample Records)")
print("=" * 70)

pd.set_option('display.max_colwidth', 60)
display_cols = ['split', 'doc_id', 'image_file', 'question', 'answer', 'reasoning_type']
df[display_cols].head(20)

 CSV Preview (Sample Records)


,split,doc_id,image_file,question,answer,reasoning_type
0,train,10002,10002.png,What is the 8?,30–Prcscntation - Pridc room at 601 Reconstituted Tobacc...,lookup
1,train,10004,10004.png,What is the 8?,30–Prcscntation - Pridc room at 601 Reconstituted Tobacc...,lookup
2,train,10005,10005.png,What is the main topic or content discussed in this sect...,This Job Assignment Addcndum is made as of thc 15th day ...,comprehension
3,train,10007,10007.png,What is the main topic or content discussed in this sect...,This Job Assignment Addcndum is made as of thc 15th day ...,comprehension
4,train,10008,10008.png,What is the main topic or content discussed in this sect...,This Job Assignment Addcndum is made as of thc 15th day ...,comprehension
5,train,1001,1001.png,What is the main topic or content discussed in this sect...,A motion has been filed to add Takeda Pharmaceuticals No...,comprehension
6,train,10010,10010.png,What is the main topic or content discussed in this sect...,"This Agreement, effective the 1st of January 1997, is en...",comprehension
7,train,10013,10013.png,What is the main topic or content discussed in this sect...,7. Sunoco will not allow supplemental fixtures or displa...,comprehension
8,train,10015,10015.png,What is the main topic or content discussed in this sect...,7. Sunoco will not allow supplemental fixtures or displa...,comprehension
9,train,1002,1002.png,What is the https?,//www.industrydocuments.ucsf.edu/docs/lzjf0226,lookup


In [11]:
# Detailed sample
print("=" * 70)
print(" Detailed Sample QA Pairs")
print("=" * 70)

for i, row in df.head(5).iterrows():
    print(f"\n{'─'*60}")
    print(f" Doc: {row['doc_id']} ({row['split']})")
    print(f" Image: {row['image_file']}")
    print(f" Question: {row['question']}")
    print(f" Answer: {row['answer'][:200]}..." if len(str(row['answer'])) > 200 else f" Answer: {row['answer']}")
    print(f" Type: {row['reasoning_type']}")

 Detailed Sample QA Pairs

────────────────────────────────────────────────────────────
 Doc: 10002 (train)
 Image: 10002.png
 Question: What is the 8?
 Answer: 30–Prcscntation - Pridc room at 601 Reconstituted Tobacco, Mr. Watson Dufour, Direcior, Product and Process Development, and Gregory Fcron, Salcs Manager.
 Type: lookup

────────────────────────────────────────────────────────────
 Doc: 10004 (train)
 Image: 10004.png
 Question: What is the 8?
 Answer: 30–Prcscntation - Pridc room at 601 Reconstituted Tobacco, Mr. Watson Dufour, Direcior, Product and Process Development, and Gregory Fcron, Salcs Manager.
 Type: lookup

────────────────────────────────────────────────────────────
 Doc: 10005 (train)
 Image: 10005.png
 Question: What is the main topic or content discussed in this section?
 Answer: This Job Assignment Addcndum is made as of thc 15th day of Sepicmber, 1998 and is cxccuted pursuant to and in accordancc with thc prcviously cxecutcd Mastcr contract bctwccn R....
 Type

## 📝 Pipeline Summary

### Output Files:
- **Graph JSONs**: `output/full_pipeline/{train,validation,test}/*.json`
- **Combined CSV**: `output/qa_csv/docvqa_qas_all_{timestamp}.csv`
- **Split CSVs**: `output/qa_csv/docvqa_qas_{split}_{timestamp}.csv`
- **Statistics**: `output/qa_csv/full_pipeline_stats_{timestamp}.json`

### CSV Columns:
| Column | Description |
|--------|-------------|
| `split` | train/validation/test |
| `doc_id` | Document ID (image stem) |
| `image_file` | Source image filename |
| `qa_id` | Unique QA ID |
| `question` | Generated question |
| `answer` | Generated answer |
| `reasoning_type` | lookup/inference/comparison/etc |
| `reasoning_explanation` | Explanation of reasoning |
| `evidence_region_ids` | JSON array of node IDs |
| `evidence_quotes` | JSON array of evidence text |